# Game Tree Search

We start with defining the abstract class `Game`, for turn-taking *n*-player games. We rely on, but do not define yet, the concept of a `state` of the game; we'll see later how individual games define states. For now, all we require is that a state has a `state.to_move` attribute, which gives the name of the player whose turn it is. ("Name" will be something like `'X'` or `'O'` for tic-tac-toe.)

We also define `play_game`, which takes a game and a dictionary of  `{player_name: strategy_function}` pairs, and plays out the game, on each turn checking `state.to_move` to see whose turn it is, and then getting the strategy function for that player and applying it to the game and the state to get a move.

In [18]:
from collections.abc import Callable
import functools
import random
import math

In [19]:
class Game:
    """A game is similar to a problem, but it has a terminal test instead of
    a goal test, and a utility for each terminal state. To create a game,
    subclass this class and implement `actions`, `result`, `is_terminal`,
    and `utility`. You will also need to set the .initial attribute to the
    initial state; this can be done in the constructor."""

    def __init__(self, initial):
        self.initial = initial

    def actions(self, state):
        """Return a collection of the allowable moves from this state."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def is_terminal(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)

    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError


def play_game(game: Game, strategies: dict[str, Callable], verbose: bool = False):
    """Play a turn-taking game. `strategies` is a {player_name: function} dict,
    where function(state, game) is used to get the player's move."""
    state = game.initial
    while not game.is_terminal(state):
        player = state.to_move
        move = strategies[player](game, state)
        state = game.result(state, move)
        if verbose:
            print(f"Player '{player}' move: {move}")
            print(state)
    return state

# Minimax-Based Game Search Algorithms

We will define several game search algorithms. Each takes two inputs, the game we are playing and the current state of the game, and returns a a `(value, move)` pair, where `value` is the utility that the algorithm computes for the player whose turn it is to move, and `move` is the move itself.

First we define `minimax_search`, which exhaustively searches the game tree to find an optimal move (assuming both players play optimally), and `alphabeta_search`, which does the same computation, but prunes parts of the tree that could not possibly have an affect on the optimnal move.  

In [20]:
def minimax_search(game: Game, state):
    """Search game tree to determine best move; return (value, move) pair."""

    player = state.to_move

    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))
            if v2 < v:
                v, move = v2, a
        return v, move

    return max_value(state)

infinity = math.inf

def alphabeta_search(game: Game, state):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    player = state.to_move

    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    def min_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity)

# A Simple Game: Tic-Tac-Toe

We have the notion of an abstract game, we have some search functions; now it is time to define a real game; a simple one, tic-tac-toe. Moves are `(x, y)` pairs denoting squares, where `(0, 0)` is the top left, and `(2, 2)` is the bottom right (on a board of size `height=width=3`).

States in tic-tac-toe (and other games) will be represented as a `Board`, which is a python `dict` object with the  some additional properties:
- `.width` and `.height` to give the size of the board (both 3 in tic-tac-toe, but other numbers in related games).
- `.to_move` to name the player whose move it is.
- `.utility` to have the current utility value.

The dictonary will consist of `{(x, y): player}` pairs, for example `{(0, 0): 'X', (1, 1): 'O'}` might be the state of the board after two moves. If no pair is defined for a specific location, that cell is considered blank.

The `Board` class has the `[]` operator defined, which returns the `player` for the assigned squares, `empty` for the ones with no assignment but that are within the `width` × `height` boundaries, or `off` otherwise. The class has a `__hash__` method, so instances can be stored in hash tables.

In [21]:
class Board(dict):
    """A board is a dict of {(x, y): player} entries, where player is 'X' or 'O', with
    additional parameters such as the player to move and a cached utility value."""

    empty = '.'
    off = '#'

    def __init__(self, width: int, height: int, to_move: str, utility: int = 0):
        self.width = width
        self.height = height
        self.to_move = to_move
        self.utility = utility

    # Override class dict's [] operator, so it doesn't raise an exception
    # when the specified key is missing
    def __getitem__(self, key: tuple[int, int]):
        x, y = key
        if key in self:
            return super().__getitem__(key)
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        return self.off

    def __hash__(self):
        return hash(tuple(sorted(self.items()))) + hash((self.width, self.height, self.to_move, self.utility))

    def __repr__(self):
        def row(y):
            return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) + '\n'

In [22]:
class TicTacToe(Game):
    """Play TicTacToe on an `height` by `width` board, needing `k` in a row to win.
    'X' plays first against 'O'."""

    def __init__(self, height: int = 3, width: int = 3, k: int = 3):
        super().__init__(initial=Board(height=height, width=width, to_move='X'))
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.k = k # k in a row

    def actions(self, board: Board):
        """Legal moves are any square not yet taken."""
        return self.squares - set(board)

    def result(self, board: Board, square: tuple[int, int]):
        """Place a marker for current player on square."""
        player = board.to_move

        next_board = Board(board.width, board.height, to_move=('O' if player=='X' else 'X'))
        next_board.update(board) # Add all the full cells to the new state
        next_board[square] = player # Make the move of the current player

        win = self.k_in_row(next_board, square, player)
        next_board.utility = (0 if not win else 1 if player == 'X' else -1)

        return next_board

    def utility(self, board: Board, player: str):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board: Board):
        """A board is a terminal state if it is won or there are no empty squares."""
        return board.utility != 0 or len(self.squares) == len(board)

    def display(self, board: Board):
        print(board)

    def k_in_row(self, board: Board, square: tuple[int, int], player: str):
        """True if player has k pieces in a line through square."""
        def in_row(x, y, dx, dy):
            if board[x, y] != player:
                return 0
            return 1 + in_row(x + dx, y + dy, dx, dy)

        # (0, 1)  -> Move on the column (dy == 1)
        # (1, 0)  -> Move on the row (dx == 1)
        # (1, 1)  -> Move on the main diagonal (dx == dy == 1)
        # (1, -1) -> Move on the secondary diagonal (dx == 1 and dy == -1)

        x, y = square
        for dx, dy in ((0, 1), (1, 0), (1, 1), (1, -1)):
            if in_row(x, y, dx, dy) + in_row(x, y, -dx, -dy) - 1 >= self.k:
                return True
        return False

def show_results(state: Board):
    print("Final state:", state, sep='\n')
    print(f"Final utility is {state.utility}")
    if state.utility == 0:
        print("The game is a tie!")
    else:
        print(f"The winner is player {'X' if state.utility == 1 else 'O'}!")

# Players

We need an interface for players. I'll represent a player as a `callable` that will be passed two arguments: `(game, state)` and will return a `move`.
The function `player` creates a player out of a search algorithm, but you can create your own players as functions, as is done with `random_player` below:

In [30]:
def random_player(game: Game, state):
    return random.choice(list(game.actions(state)))

def ai_player(search_algorithm: Callable):
    """A game player who uses the specified search algorithm"""
    return lambda game, state: search_algorithm(game, state)[1]

def human_player(_: Game, state: Board):
    print(f"Player '{state.to_move}' please make your move!")
    while True:
        try:
            row = int(input("Enter row: "))
            col = int(input("Enter col: "))
            if state[col, row] == Board.empty: break
        except ValueError:
            pass
        print("Invalid move, please try again...")
    return (col, row)

# Playing a Game

We're ready to play a game. I'll set up a match between a `random_player` (who chooses randomly from the legal moves) and a `player(alphabeta_search)` (who makes the optimal alpha-beta move; practical for tic-tac-toe, but not for large games). The `player(alphabeta_search)` will never lose, but if `random_player` is lucky, it will be a tie.

In [ ]:
state = play_game(TicTacToe(), dict(X=random_player, O=ai_player(alphabeta_search)), verbose=True)
show_results(state)

The alpha-beta player will never lose, but sometimes the random player can stumble into a draw. When two optimal (alpha-beta or minimax) players compete, it will always be a draw:

In [ ]:
state = play_game(TicTacToe(), dict(X=ai_player(alphabeta_search), O=ai_player(minimax_search)), verbose=True)
show_results(state)

# Connect Four

Connect Four is a variant of tic-tac-toe, played on a larger (7 x 6) board, and with the restriction that in any column you can only play in the lowest empty square in the column.

In [34]:
class ConnectFour(TicTacToe):

    def __init__(self):
        super().__init__(width=7, height=6, k=4)

    def actions(self, board: Board):
        """In each column you can play only the lowest empty square in the column."""
        return {(x, y) for x, y in self.squares - set(board)
                if y == board.height - 1 or (x, y + 1) in board}

In [ ]:
state = play_game(ConnectFour(), dict(X=random_player, O=random_player), verbose=True)
show_results(state)

# Transposition Tables

By treating the game tree as a tree, we can arrive at the same state through different paths, and end up duplicating effort. In state-space search, we kept a table of `reached` states to prevent this. For game-tree search, we can achieve the same effect by applying the `@cache` decorator to the `min_value` and `max_value` functions. We'll use the suffix `_tt` to indicate a function that uses these transisiton tables.

In [36]:
cache = functools.lru_cache(10**6)

def minimax_search_cached(game, state):
    """Search game to determine best move; return (value, move) pair."""

    player = state.to_move

    @cache
    def max_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a))
            if v2 > v:
                v, move = v2, a
        return v, move

    @cache
    def min_value(state):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a))
            if v2 < v:
                v, move = v2, a
        return v, move

    return max_value(state)

For alpha-beta search, we can still use a cache, but it should be based just on the state, not on whatever values alpha and beta have.

In [37]:
def cache1(function):
    "Like lru_cache(None), but only considers the first argument of function."
    cache = {}
    def wrapped(x, *args):
        if x not in cache:
            cache[x] = function(x, *args)
        return cache[x]
    return wrapped

def alphabeta_search_cached(game, state):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    player = state.to_move

    @cache1
    def max_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    @cache1
    def min_value(state, alpha, beta):
        if game.is_terminal(state):
            return game.utility(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity)

In [ ]:
%time state = play_game(TicTacToe(), {'X': ai_player(alphabeta_search_cached), 'O': ai_player(minimax_search_cached)})
show_results(state)

In [ ]:
%time state = play_game(TicTacToe(), {'X': ai_player(alphabeta_search), 'O': ai_player(minimax_search)})
show_results(state)

# Heuristic Cutoffs

In [40]:
def cutoff_depth(d):
    """A cutoff function that searches to depth d."""
    return lambda game, state, depth: depth > d

def h_alphabeta_search(game, state, cutoff=cutoff_depth(6), h=lambda s, p: 0):
    """Search game to determine best action; use alpha-beta pruning.
    As in [Figure 5.7], this version searches all the way to the leaves."""

    player = state.to_move

    @cache1
    def max_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = -infinity, None
        for a in game.actions(state):
            v2, _ = min_value(game.result(state, a), alpha, beta, depth+1)
            if v2 > v:
                v, move = v2, a
                alpha = max(alpha, v)
            if v >= beta:
                return v, move
        return v, move

    @cache1
    def min_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if cutoff(game, state, depth):
            return h(state, player), None
        v, move = +infinity, None
        for a in game.actions(state):
            v2, _ = max_value(game.result(state, a), alpha, beta, depth + 1)
            if v2 < v:
                v, move = v2, a
                beta = min(beta, v)
            if v <= alpha:
                return v, move
        return v, move

    return max_value(state, -infinity, +infinity, 0)

In [ ]:
%time state = play_game(TicTacToe(), {'X': ai_player(h_alphabeta_search), 'O': ai_player(h_alphabeta_search)})
show_results(state)

In [ ]:
%time state = play_game(ConnectFour(), {'X':ai_player(h_alphabeta_search), 'O':ai_player(h_alphabeta_search)}, verbose=True)
show_results(state)

In [ ]:
# Normal/Cache/Heuristic approach comparison...
print("Normal alphabeta_search")
%time play_game(TicTacToe(), {'X':ai_player(alphabeta_search), 'O': random_player})
print("\nCached alphabeta_search")
%time play_game(TicTacToe(), {'X':ai_player(alphabeta_search_cached), 'O': random_player})
print("\nHeuristic alphabeta_search")
%time play_game(TicTacToe(), {'X':ai_player(h_alphabeta_search), 'O': random_player})
print()